# ClinicalRecall — A100 GPU Ingestion on Colab

**Prerequisites:**
- `data/` folder uploaded to Google Drive at `MyDrive/ClinicalRecall/data/`
- Contains: `fhir/` (5,770 JSON files), `clinician_notes/` (5,770 JSON files), `mtsamples_staging.db`

**What this notebook does:**
1. Mounts Google Drive
2. Clones the repo from GitHub
3. Installs dependencies
4. Auto-tunes batch size for A100 (40GB VRAM)
5. Runs `--only synthea` ingestion
6. Downloads the resulting `chroma_db/` back to Drive

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/ClinicalRecall/data/

## 2. Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/vignesh-kumar-v/ClinIQ.git /content/ClinicalRecall

In [ ]:
%cd /content/ClinicalRecall
!git checkout data-pipeline
!git pull origin data-pipeline

In [ ]:
!pip install -q chromadb sentence-transformers torch openai python-dotenv

## 3. Link Data from Drive

In [ ]:
!rm -rf /content/ClinicalRecall/data
!ln -s /content/drive/MyDrive/ClinicalRecall/data /content/ClinicalRecall/data
!ls /content/ClinicalRecall/data/

## 4. Verify GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 5. Run Ingestion

This will auto-tune the batch size for the A100, then ingest all 5,770 patients.
Expected: ~10-15 minutes on A100.

In [ ]:
!python scripts/ingest_chromadb_gpu.py --only synthea

## 6. Verify Output

In [ ]:
import sqlite3
conn = sqlite3.connect('chroma_db/chroma.sqlite3')

cols = conn.execute("SELECT name FROM collections").fetchall()
print(f"Collections: {[c[0] for c in cols]}")

total = conn.execute("SELECT COUNT(*) FROM embeddings").fetchone()[0]
print(f"Total embeddings: {total:,}")

patients = conn.execute('''
    SELECT COUNT(DISTINCT em.string_value) 
    FROM embedding_metadata em 
    WHERE em.key = 'patient_id'
''').fetchone()[0]
print(f"Unique patients: {patients:,}")

# Check encounter context
with_enc = conn.execute('''
    SELECT COUNT(DISTINCT em.id) 
    FROM embedding_metadata em 
    WHERE em.key = 'encounter_type' AND em.string_value != ''
''').fetchone()[0]
print(f"Chunks with encounter_type: {with_enc:,} ({with_enc/total*100:.1f}%)")

# Check metadata keys
keys = conn.execute("SELECT DISTINCT key FROM embedding_metadata").fetchall()
print(f"Metadata keys: {sorted([k[0] for k in keys])}")

conn.close()

In [ ]:
!du -sh chroma_db/

## 7. Copy chroma_db to Drive

In [ ]:
!cp -r /content/ClinicalRecall/chroma_db /content/drive/MyDrive/ClinicalRecall/chroma_db_a100
print("Done! chroma_db copied to Drive as chroma_db_a100")

## Optional: Run Full Pipeline (MTSamples + Logs too)

Uncomment and run if you want the complete ingestion:

In [ ]:
# !python scripts/ingest_chromadb_gpu.py --reset